# Модуль 07 · Урок 2. REST API, вложенный JSON и нормализация

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Работа с API — отдельный этап извлечения
REST API обычно возвращает JSON-объекты или массивы объектов. В полноценном приложении HTTP-клиент отвечает за тайм-ауты, статус ответа, повторы, авторизацию, пагинацию и лимиты запросов; pandas начинает работу **после того**, как данные безопасно получены. В учебнике мы обрабатываем сохранённые JSON-ответы локально — задания не зависят от доступности сервера.

In [ ]:
import pandas as pd
api_response = [{'user_id':1,'profile':{'city':'Томск'}},{'user_id':2,'profile':{'city':'Омск'}}]
print(pd.json_normalize(api_response))

## 2. json_normalize и вложенные структуры
`pd.json_normalize()` превращает вложенные словари в плоские колонки (`profile.city`). Для вложенных списков объектов часто проще применить `explode('lines')`, а затем `pd.json_normalize` к каждому элементу. Учитывайте смысл: одна исходная запись заказа после разворачивания нескольких позиций превращается в несколько строк. Если считать число заказов по строкам позиций, результат будет завышен.

In [ ]:
shipments = [{'id':'A','lines':[{'code':'P','qty':1},{'code':'Q','qty':3}]}]
flat=pd.DataFrame(shipments).explode('lines',ignore_index=True)
print(pd.concat([flat[['id']],pd.json_normalize(flat['lines'])],axis=1))

## 3. Пагинация и дедупликация
API часто отдаёт данные по страницам. После загрузки каждой страницы соберите список записей и объедините их через `concat(...,ignore_index=True)`. Страницы могут перекрываться при обновлении данных на сервере, поэтому при необходимости проверяйте ключ и убирайте дубликаты **по бизнес-правилу**. Запрашивать новые страницы до бесконечности нельзя: требуется курсор, флаг окончания или проверка отсутствия данных.

## 4. Контракты и отсутствие полей
Не все JSON-объекты содержат все ключи. В DataFrame пропущенные поля обычно становятся NA. Нельзя надеяться, что поле обязательно существует, если так было в первом примере ответа. Для устойчивости: укажите ожидаемую схему, проверьте нужные колонки и типы, отделите некорректные записи.

**Проверьте себя:** почему после explode растёт число строк? Как посчитать уникальные заказы после нормализации? Что делать с отсутствующим ключом API?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m7_06 · flatten users

Нормализуйте вложенные JSON-объекты списка через `pd.json_normalize`. Получите DataFrame, поля пользователя должны стать `user.country`.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `rows` — список вложенных словарей, содержащих данные API; тип: `list`.


**Выполнение:** используйте готовые переменные `rows`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_06: используются переменные rows

rows = [{'id': 'a',
  'user': {'country': 'RU'},
  'lines': [{'sku': 'S1', 'qty': 2}, {'sku': 'S2', 'qty': 1}]},
 {'id': 'b', 'user': {'country': 'KZ'}, 'lines': [{'sku': 'S3', 'qty': 4}]}]

print("rows =", rows)


# Снимок правильного результата для автопроверки.
_expected_m7_06 = expected_for("m7_06", rows=rows)


In [ ]:
# m7_06: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: rows
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 3 столбца; колонки: `id`, `lines`, `user.country`.

```text
  id            lines user.country
0  a  [{'sku': 'S1...           RU
1  b  [{'sku': 'S3...           KZ
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_06", result, _expected_m7_06)

### m7_07 · orders lines

Разполучите каждый элемент lines в отдельную строку. Результат — DataFrame `order_id`,`sku`,`qty` в исходном порядке заказов и строк.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `rows` — список вложенных словарей, содержащих данные API; тип: `list`.


**Выполнение:** используйте готовые переменные `rows`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_07: используются переменные rows

rows = [{'id': 'a',
  'user': {'country': 'RU'},
  'lines': [{'sku': 'S1', 'qty': 2}, {'sku': 'S2', 'qty': 1}]},
 {'id': 'b', 'user': {'country': 'KZ'}, 'lines': [{'sku': 'S3', 'qty': 4}]}]

print("rows =", rows)


# Снимок правильного результата для автопроверки.
_expected_m7_07 = expected_for("m7_07", rows=rows)


In [ ]:
# m7_07: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: rows
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 3 столбца; колонки: `order_id`, `sku`, `qty`.

```text
  order_id sku  qty
0        a  S1    2
1        a  S2    1
2        b  S3    4
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_07", result, _expected_m7_07)

### m7_08 · country counts

Получите Series числа заказов по `user.country`, отсортированную по названию страны; серия должна называться orders.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `rows` — список вложенных словарей, содержащих данные API; тип: `list`.


**Выполнение:** используйте готовые переменные `rows`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_08: используются переменные rows

rows = [{'id': 'a',
  'user': {'country': 'RU'},
  'lines': [{'sku': 'S1', 'qty': 2}, {'sku': 'S2', 'qty': 1}]},
 {'id': 'b', 'user': {'country': 'KZ'}, 'lines': [{'sku': 'S3', 'qty': 4}]}]

print("rows =", rows)


# Снимок правильного результата для автопроверки.
_expected_m7_08 = expected_for("m7_08", rows=rows)


In [ ]:
# m7_08: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: rows
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 2 значений; имя серии: `orders`.

```text
user.country
KZ    1
RU    1
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_08", result, _expected_m7_08)

### m7_09 · paginate

Получите список DataFrame-страниц: объедините их вертикально, удалите дубликаты по `order_id` (оставьте последнее появление), сбросьте индекс.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `pages` — список таблиц (`list[pd.DataFrame]`); тип: `list`.


**Выполнение:** используйте готовые переменные `pages`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_09: используются переменные pages

pages_part0 = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3'],
    'client_id': [1, 2, 1],
    'city': ['Москва', 'Тверь', 'Москва'],
    'category': ['чай', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0],
    'qty': [2, 1, 4],
    'discount': [0, 10, 5],
    'status': ['paid', 'new', 'paid'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно'],
})
pages_part1 = pd.DataFrame({
    'order_id': ['demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 3, 2, 4],
    'city': ['Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'книги', 'кофе', 'чай'],
    'price': [97.0, 317.0, 57.0, 147.0],
    'qty': [4, 1, 3, 0],
    'discount': [5, 0, 20, 0],
    'status': ['paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['Срочно', 'подарок', '', None],
})
pages = [pages_part0, pages_part1]

print("pages =", pages)


# Снимок правильного результата для автопроверки.
_expected_m7_09 = expected_for("m7_09", pages=pages)


In [ ]:
# m7_09: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: pages
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment
0  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...
1  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка
2  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно
3  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок
4  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                 
5  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_09", result, _expected_m7_09)

### m7_10 · lines revenue

Для вложенных JSON-заказов получите Series с общей суммой qty*price по order_id в порядке сортировки ключей. Цены SKU находятся в словаре price_map.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `rows` — список вложенных словарей, содержащих данные API; тип: `list`.
- `price_map` — словарь цен по артикулам; тип: `dict`.


**Выполнение:** используйте готовые переменные `rows`, `price_map`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_10: используются переменные rows, price_map

rows = [{'id': 'a',
  'user': {'country': 'RU'},
  'lines': [{'sku': 'S1', 'qty': 2}, {'sku': 'S2', 'qty': 1}]},
 {'id': 'b', 'user': {'country': 'KZ'}, 'lines': [{'sku': 'S3', 'qty': 4}]}]

price_map = {'S1': 10, 'S2': 20, 'S3': 5}

print("rows =", rows)

print("price_map =", price_map)


# Снимок правильного результата для автопроверки.
_expected_m7_10 = expected_for("m7_10", rows=rows, price_map=price_map)


In [ ]:
# m7_10: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: rows, price_map
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 2 значений; имя серии: `total`.

```text
order_id
a    40
b    20
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_10", result, _expected_m7_10)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()